# Machine Learning Classification: Comparative Analysis of SVM, Decision Tree, KNN and Naïve Bayes

**Student Name:** Ashmit Amit Budhe  
**PRN:** 202401040047  
**Batch:** A1  
**Roll No.:** 12  

**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset (Scikit-learn)  
**Platform:** Google Colab  
**Objective:** Develop four classification models and compare them using Accuracy, Precision, Recall, F1-score and Log Loss, supported by graphs and a comparative discussion.

---

## 1. Aim

To implement Support Vector Machine (SVM), Decision Tree, K-Nearest Neighbors (KNN), and Gaussian Naïve Bayes classification models using the Breast Cancer Wisconsin dataset, evaluate their performance using relevant metrics, and discuss their suitability for different classification tasks.

## 2. Dataset description

The Breast Cancer Wisconsin Diagnostic dataset contains 569 observations and 30 numerical features computed from digitized images of breast mass samples. The target has two classes:

- **0 – Malignant**
- **1 – Benign**

This is a binary classification problem. The dataset is included with Scikit-learn, so no separate CSV download is required.

## 3. Algorithms used

1. **SVM:** Finds a decision boundary that separates classes, aiming to maximize the margin.
2. **Decision Tree:** Uses feature-based decision rules arranged in a tree structure.
3. **KNN:** Predicts a class based on the classes of nearby training observations.
4. **Gaussian Naïve Bayes:** Uses Bayes' theorem and models continuous features with Gaussian distributions.

## 4. Evaluation metrics

- **Accuracy:** Fraction of all predictions that are correct.
- **Precision (Malignant class):** Of the samples predicted malignant, the fraction that are actually malignant.
- **Recall (Malignant class):** Of all actual malignant samples, the fraction correctly identified.
- **F1-score (Malignant class):** Harmonic mean of malignant-class precision and recall.
- **Log Loss:** Evaluates the quality of predicted probabilities; lower is better.

For precision, recall and F1-score, malignant (class 0) is treated as the positive class. This makes malignant-class recall particularly useful to inspect. These are educational classification results, not medical diagnoses.


## 5. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 6. Load the dataset and inspect it

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="Target")

print("Dataset loaded successfully.")
print("Number of records:", X.shape[0])
print("Number of features:", X.shape[1])
print("\nTarget mapping:", {0: "Malignant", 1: "Benign"})
print("\nClass counts:")
print(y.value_counts().sort_index().rename(index={0: "Malignant", 1: "Benign"}))
print("\nFirst five records:")
display(X.head())

In [ ]:
print("Dataset information:")
display(X.info())

print("\nMissing values in the feature dataset:", X.isnull().sum().sum())
print("\nStatistical summary:")
display(X.describe().T)

## 7. Explore and visualize the dataset

In [ ]:
class_counts = y.map({0: "Malignant", 1: "Benign"}).value_counts()

plt.figure(figsize=(7, 5))
sns.countplot(x=y.map({0: "Malignant", 1: "Benign"}),
              order=["Malignant", "Benign"])
plt.title("Distribution of Breast Cancer Classes")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

print(class_counts)

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Dataset Features")
plt.tight_layout()
plt.show()

## 8. Split the dataset into training and testing sets

The data is split into 80% training and 20% testing. `stratify=y` preserves the approximate class proportions in both sets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 9. Build and train the four classification models

Feature scaling is used for SVM and KNN because they are sensitive to feature magnitudes. Scaling is placed inside a pipeline so that the scaler is fitted only on training data, preventing data leakage. Decision Tree and Gaussian Naïve Bayes are trained on the original feature values.

The SVM uses an RBF kernel and probability estimation so that Log Loss and ROC curves can be calculated.

In [ ]:
models = {
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", probability=True, random_state=42)
    ),
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini", max_depth=5, random_state=42
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),
    "Naive Bayes": GaussianNB()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully.")

## 10. Generate predictions and evaluate performance

All metrics are calculated using the same held-out test set. Precision, recall and F1-score use malignant (class 0) as the positive class. Log Loss uses the predicted class probabilities.

In [ ]:
predictions = {}
probabilities = {}
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    predictions[name] = y_pred
    probabilities[name] = y_proba

    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Malignant)": precision_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "Recall (Malignant)": recall_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "F1-score (Malignant)": f1_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "Log Loss": log_loss(y_test, y_proba, labels=[0, 1])
    })

results_df = pd.DataFrame(results).set_index("Algorithm")
print("Comparative performance table:")
display(results_df.round(4))

## 11. Classification reports

The report provides precision, recall and F1-score separately for malignant and benign classes, along with the number of test samples (support).

In [ ]:
for name in models:
    print("\n" + "=" * 65)
    print("CLASSIFICATION REPORT:", name)
    print("=" * 65)
    print(classification_report(
        y_test,
        predictions[name],
        labels=[0, 1],
        target_names=["Malignant", "Benign"],
        zero_division=0
    ))

## 12. Confusion matrices

Rows represent actual classes and columns represent predicted classes. The diagonal cells are correct predictions; off-diagonal cells are errors. For malignant-class recall, pay attention to malignant samples predicted as benign.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, y_pred) in zip(axes.flatten(), predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Malignant", "Benign"],
        yticklabels=["Malignant", "Benign"],
        ax=ax
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")

plt.suptitle("Confusion Matrix Comparison", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

## 13. Compare Accuracy, Precision, Recall and F1-score

In [ ]:
metric_columns = [
    "Accuracy",
    "Precision (Malignant)",
    "Recall (Malignant)",
    "F1-score (Malignant)"
]

results_df[metric_columns].plot(
    kind="bar", figsize=(12, 6), ylim=(0, 1.05), width=0.8
)
plt.title("Classification Algorithm Performance Comparison")
plt.xlabel("Algorithm")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 14. Compare Log Loss

Unlike the other scores, lower Log Loss is better. It evaluates the predicted probabilities, not just the final predicted class.

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    x=results_df.index,
    y=results_df["Log Loss"]
)
plt.title("Log Loss Comparison (Lower is Better)")
plt.xlabel("Algorithm")
plt.ylabel("Log Loss")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 15. ROC curves and AUC

The ROC curve shows the true-positive rate against the false-positive rate at different thresholds. Here, malignant (class 0) is treated as the positive class. AUC closer to 1 indicates stronger class-ranking performance.

In [ ]:
plt.figure(figsize=(9, 7))
y_test_malignant = (y_test == 0).astype(int)

for name in models:
    # Column 0 corresponds to class 0 (Malignant).
    malignant_probability = probabilities[name][:, 0]
    fpr, tpr, thresholds = roc_curve(
        y_test_malignant, malignant_probability
    )
    auc_value = roc_auc_score(
        y_test_malignant, malignant_probability
    )
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Visualize the Decision Tree

In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(
    models["Decision Tree"],
    feature_names=data.feature_names,
    class_names=["Malignant", "Benign"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree Structure")
plt.tight_layout()
plt.show()

## 17. Identify the best model for each metric

The best algorithm can differ by metric. The following cell uses the measured test-set results rather than assuming that one algorithm is best for every task.

In [ ]:
print("BEST MODEL BY METRIC")
print("-" * 50)

for metric in metric_columns:
    best_algorithm = results_df[metric].idxmax()
    best_score = results_df.loc[best_algorithm, metric]
    print(f"Highest {metric}: {best_algorithm} ({best_score:.4f})")

best_loss_algorithm = results_df["Log Loss"].idxmin()
best_loss = results_df.loc[best_loss_algorithm, "Log Loss"]
print(f"Lowest Log Loss: {best_loss_algorithm} ({best_loss:.4f})")

best_auc = {}
for name in models:
    best_auc[name] = roc_auc_score(
        y_test_malignant, probabilities[name][:, 0]
    )
print("\nROC-AUC scores:")
for name, score in best_auc.items():
    print(f"{name}: {score:.4f}")

## 18. Comparative analysis and suitability

Use the measured results above to support the discussion. General characteristics of the algorithms are summarized below.

In [ ]:
comparison = pd.DataFrame({
    "Algorithm": ["SVM", "Decision Tree", "KNN", "Naive Bayes"],
    "Advantages": [
        "Can model complex decision boundaries; effective with scaled high-dimensional data.",
        "Easy to interpret and can represent decisions as rules.",
        "Simple method based on nearby training examples; can model nonlinear boundaries.",
        "Fast to train and predict; provides class probabilities."
    ],
    "Limitations": [
        "Needs feature scaling and parameter/kernel selection; probability estimation can add cost.",
        "Can overfit without constraints and may be unstable to small data changes.",
        "Prediction can be expensive on large datasets and is sensitive to scaling and choice of k.",
        "Conditional-independence assumption may be unrealistic for correlated features."
    ],
    "Suitable tasks": [
        "High-dimensional data and tasks with complex class boundaries.",
        "Interpretable, rule-based decisions and smaller structured datasets.",
        "Small or medium datasets where similar observations have similar labels.",
        "Fast baseline classification and some text-classification tasks."
    ]
})
display(comparison)

### Discussion of findings

- **SVM:** A useful choice when a separating boundary is complex and feature scaling is applied. Its performance depends on the kernel and parameters.
- **Decision Tree:** Useful when interpretability is important because its decisions can be inspected. Limiting tree depth can help reduce overfitting.
- **KNN:** A straightforward distance-based method. It can work well when nearby observations tend to share a class, but it requires scaling and can be slower at prediction time.
- **Gaussian Naïve Bayes:** A fast probabilistic baseline for continuous features. Its conditional-independence assumption can limit performance when features are strongly related.

There is no universally best classifier. The preferred model depends on the metric that matters, the cost of different errors, the dataset size and structure, interpretability requirements, and prediction-time constraints. For this dataset, inspect malignant-class recall and the confusion matrices in addition to accuracy.

## 19. Automatically generated result summary

In [ ]:
best_accuracy_model = results_df["Accuracy"].idxmax()
best_precision_model = results_df["Precision (Malignant)"].idxmax()
best_recall_model = results_df["Recall (Malignant)"].idxmax()
best_f1_model = results_df["F1-score (Malignant)"].idxmax()
best_logloss_model = results_df["Log Loss"].idxmin()

print("RESULT SUMMARY")
print("-" * 55)
print(f"Highest Accuracy: {best_accuracy_model} "
      f"({results_df.loc[best_accuracy_model, 'Accuracy']:.4f})")
print(f"Highest Malignant Precision: {best_precision_model} "
      f"({results_df.loc[best_precision_model, 'Precision (Malignant)']:.4f})")
print(f"Highest Malignant Recall: {best_recall_model} "
      f"({results_df.loc[best_recall_model, 'Recall (Malignant)']:.4f})")
print(f"Highest Malignant F1-score: {best_f1_model} "
      f"({results_df.loc[best_f1_model, 'F1-score (Malignant)']:.4f})")
print(f"Lowest Log Loss: {best_logloss_model} "
      f"({results_df.loc[best_logloss_model, 'Log Loss']:.4f})")

print("\nFull results:")
display(results_df.round(4))

## 20. Conclusion

In this experiment, four supervised machine learning classification algorithms—Support Vector Machine (SVM), Decision Tree, K-Nearest Neighbors (KNN), and Gaussian Naïve Bayes—were implemented using the Breast Cancer Wisconsin dataset.

The dataset was divided into training and testing subsets using an 80:20 split. The models were evaluated using Accuracy, Precision, Recall, F1-score and Log Loss. Confusion matrices, metric-comparison graphs, ROC curves and a Decision Tree visualization were also generated.

The algorithms have different strengths. SVM can model complex decision boundaries, Decision Tree offers interpretability, KNN classifies based on nearby observations, and Naïve Bayes provides a fast probabilistic approach. The best-performing algorithm depends on the measured results and the evaluation metric of interest.

**Final experimental finding:** Refer to the automatically generated result summary in Section 19 and insert the measured algorithm names and scores into your submitted report. Do not claim that one algorithm is best unless the results support that conclusion.

This experiment demonstrates how classification algorithms can be implemented, evaluated and compared for a binary classification task. The predictions are for academic demonstration only and must not be used for medical decision-making.

## 21. References

1. Scikit-learn Developers, “Breast Cancer Wisconsin (Diagnostic) Dataset,” Scikit-learn documentation: https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset
2. Scikit-learn Developers, “Supervised learning,” https://scikit-learn.org/stable/supervised_learning.html
3. Scikit-learn Developers, “Model evaluation: quantifying the quality of predictions,” https://scikit-learn.org/stable/modules/model_evaluation.html
4. Google Colab, https://colab.research.google.com/

---

**End of Assignment**  
**Student:** Ashmit Amit Budhe  
**PRN:** 202401040047  
**Batch:** A1  
**Roll No.:** 12
